In [4]:
import pandas as pd
import numpy as np

In [6]:
from google.colab import drive
# Mount Google Drive
drive.mount('/content/drive')

# 1. Load dataset from Google Drive path using the standard mount path
excel_file = "/content/drive/MyDrive/20260512_FintechTrainee_CaseStudy_Dataset_FinSightLanka_F.xlsx"
df = pd.read_excel(excel_file, sheet_name='Customer_Data')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
print(f"Original row count: {len(df)}")

Original row count: 508


In [8]:
# 2. Data Cleaning & Handling Quality Issues
# Drop duplicate Customer IDs
df_clean = df.drop_duplicates(subset=['Customer_ID']).copy()

In [9]:
# Remove extreme unrealistic outliers in savings and withdrawals
df_clean = df_clean[~df_clean['Savings_Balance'].isin([999999999.0])].copy()
df_clean = df_clean[~df_clean['Monthly_Withdrawal_Avg'].isin([9999999.0, 8888888.0])].copy()

In [10]:
print(f"Cleaned unique customer count: {len(df_clean)}")

Cleaned unique customer count: 497


In [ ]:
# Check for missing values in the cleaned dataset
missing_values = df_clean.isnull().sum()
missing_only = missing_values[missing_values > 0]

if not missing_only.empty:
    print("Columns with missing values and their counts:")
    display(missing_only.to_frame(name='Missing Count'))
else:
    print("No missing values found in the cleaned dataset!")

In [19]:
# Explicitly print missing value details in df_clean
null_counts = df_clean.isnull().sum()
null_cols = null_counts[null_counts > 0]
if len(null_cols) > 0:
    print("Columns with missing values and their counts:")
    for col, val in null_cols.items():
        print(f" - {col}: {val} missing values")
else:
    print("Great news! No missing values found in the cleaned dataset (df_clean).")

Columns with missing values and their counts:
 - Full_Name: 8 missing values
 - District: 5 missing values
 - Customer_Segment: 5 missing values
 - Savings_Balance: 6 missing values
 - Monthly_Deposit_Avg: 6 missing values
 - Has_Loan: 5 missing values
 - Loan_Type: 279 missing values
 - Loan_Amount: 280 missing values
 - Loan_Tenure_Months: 279 missing values
 - Monthly_Repayment_Amount: 279 missing values
 - Loan_Repayment_Status: 277 missing values
 - Loan_Start_Date: 277 missing values
 - Outstanding_Loan_Balance: 278 missing values
 - Last_Login_Date: 167 missing values
 - KYC_Status: 7 missing values
 - FD_Amount: 354 missing values
 - Referral_Source: 247 missing values


In [12]:
# 3. Question 1 Calculations: Savings & Customer Behaviour
avg_savings_segment = df_clean.groupby('Customer_Segment')['Savings_Balance'].mean().reset_index()
avg_savings_province = df_clean.groupby('Province')['Savings_Balance'].mean().reset_index()
avg_savings_urban = df_clean.groupby('Urban_Rural')['Savings_Balance'].mean().reset_index()

In [13]:
# Age banding logic
def categorize_age(age):
    if pd.isna(age):
        return 'Unknown'
    elif age <= 30:
        return '18-30 (Young Adults)'
    elif age <= 50:
        return '31-50 (Adults)'
    else:
        return '51+ (Seniors)'

df_clean['Age_Band'] = df_clean['Age'].apply(categorize_age)
total_savings_age = df_clean.groupby('Age_Band')['Savings_Balance'].sum().reset_index()

In [14]:
# 4. Question 2 Calculations: Loan Portfolio & Risk
total_customers = len(df_clean)
loans_yes = (df_clean['Has_Loan'] == 'Yes').sum()
loan_penetration = (loans_yes / total_customers) * 100

In [15]:
# Debt-to-Savings Ratio for loan holders
loan_df = df_clean[df_clean['Has_Loan'] == 'Yes'].copy()
loan_df['Debt_Savings_Ratio'] = loan_df['Outstanding_Loan_Balance'] / loan_df['Savings_Balance'].replace(0, np.nan)
exceed_1_count = (loan_df['Debt_Savings_Ratio'] > 1.0).sum()
exposure_exceed_1 = loan_df[loan_df['Debt_Savings_Ratio'] > 1.0]['Outstanding_Loan_Balance'].sum()

In [16]:
# 5. Question 3 Calculations: Digital Engagement & Product Holdings
app_yes_savings = df_clean[df_clean['Mobile_App_User'] == 'Yes']['Savings_Balance'].mean()
app_no_savings = df_clean[df_clean['Mobile_App_User'] == 'No']['Savings_Balance'].mean()

# Cross-sell opportunity: Savings > 0, No Fixed Deposit, No Insurance
cross_sell_df = df_clean[(df_clean['Savings_Balance'] > 0) & (df_clean['Has_Fixed_Deposit'] == 'No') & (df_clean['Has_Insurance'] == 'No')]
cross_sell_count = len(cross_sell_df)
cross_sell_value = cross_sell_df['Savings_Balance'].sum()

In [18]:
# 6. Export all cleaned data and summaries into a multi-sheet Excel Workbook
output_filename = "/content/drive/MyDrive/Hiruni_Siyambalapitiya_FinSightAssessment_Workings.xlsx"
with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
    df_clean.to_excel(writer, sheet_name='Cleaned_Customer_Data', index=False)
    avg_savings_segment.to_excel(writer, sheet_name='Q1_Segment_Savings', index=False)
    avg_savings_province.to_excel(writer, sheet_name='Q1_Province_Savings', index=False)
    total_savings_age.to_excel(writer, sheet_name='Q1_AgeBand_Savings', index=False)
    loan_df.to_excel(writer, sheet_name='Q2_Loan_Analysis', index=False)
    cross_sell_df.to_excel(writer, sheet_name='Q3_CrossSell_Opportunity', index=False)

print(f"\nWorkings file successfully created and saved directly to your Google Drive as: {output_filename}")


Workings file successfully created and saved directly to your Google Drive as: /content/drive/MyDrive/Hiruni_Siyambalapitiya_FinSightAssessment_Workings.xlsx


In [21]:
from google.colab import files

# Trigger browser download for the Excel file
files.download(output_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>